# India RDD + local crack → YOLOv8 (Colab T4)

Uses your existing split:

`/content/drive/MyDrive/datasets/archive/RDD_SPLIT/{train,val,test}/{images,labels}`

**India filter:** keep files whose name starts with `India_`.

**Store filtered road set in this repo:**

`data/road/{train,val,test}/{images,labels}`

**Crack set:** anything already under `data/` except `data/road/` (for example `data/crack/`).

**Flask weights:** `municipal_ai_system/app/ai_models/best.pt`

Class IDs must stay:

| id | name |
|----|------|
| 0 | D00_longitudinal_crack |
| 1 | D10_transverse_crack |
| 2 | D20_alligator_crack |
| 3 | D40_pothole |

Runtime → Change runtime type → **T4 GPU** → Restart.


## CELL 1 — Packages + GPU


In [ ]:
%pip install -q ultralytics opencv-python-headless matplotlib pyyaml Pillow

import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CPU only. Runtime → Change runtime type → T4 GPU → Restart")


## CELL 2 — Mount Drive and locate folders


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

RDD_SPLIT = Path("/content/drive/MyDrive/datasets/archive/RDD_SPLIT")
assert RDD_SPLIT.exists(), f"Not found: {RDD_SPLIT}"

MYDRIVE = Path("/content/drive/MyDrive")
REPO_CANDIDATES = [
    MYDRIVE / "Road Inspection System",
    MYDRIVE / "Road-Inspection-System",
    MYDRIVE / "RoadInspectionSystem",
    MYDRIVE / "Desktop" / "Road Inspection System",
]
REPO = next((p for p in REPO_CANDIDATES if (p / "municipal_ai_system" / "app" / "ai_models").exists()), None)
if REPO is None:
    for child in MYDRIVE.iterdir():
        if not child.is_dir():
            continue
        hit = child / "municipal_ai_system" / "app" / "ai_models"
        if hit.exists():
            REPO = child
            break
if REPO is None:
    raise FileNotFoundError(
        "Could not find municipal_ai_system/app/ai_models on Drive. "
        "Add a shortcut to the project folder under MyDrive."
    )

DATA_ROOT = REPO / "data"
ROAD_OUT = DATA_ROOT / "road"
CRACK_ROOT = DATA_ROOT / "crack"
CRACK_SEG = REPO / "municipal_ai_system" / "data" / "Crack dataset"
AI_MODELS = REPO / "municipal_ai_system" / "app" / "ai_models"
LOCAL_INDIA = Path("/content/rdd_india_yolo")
LOCAL_COMBINED = Path("/content/combined-dataset")

print("REPO:", REPO)
print("RDD_SPLIT:", RDD_SPLIT)
print("ROAD_OUT (filtered India):", ROAD_OUT)
print("CRACK_ROOT:", CRACK_ROOT, "exists=", CRACK_ROOT.exists())
print("CRACK_SEG:", CRACK_SEG, "exists=", CRACK_SEG.exists())
print("AI_MODELS:", AI_MODELS)

for split in ("train", "val", "test"):
    img_dir = RDD_SPLIT / split / "images"
    lbl_dir = RDD_SPLIT / split / "labels"
    n_img = len([p for p in img_dir.glob("*") if p.suffix.lower() in IMAGE_EXTS]) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    print(f"{split}: images={n_img}, labels={n_lbl}")


## CELL 3 — Class map (Flask-compatible)


In [ ]:
from __future__ import annotations

import random
import shutil
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import yaml
from ultralytics import YOLO

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CLASS_TO_ID = {"D00": 0, "D10": 1, "D20": 2, "D40": 3}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}
CLASS_NAMES = {
    0: "D00_longitudinal_crack",
    1: "D10_transverse_crack",
    2: "D20_alligator_crack",
    3: "D40_pothole",
}
KEEP_CLASSES = {0, 1, 2, 3}
COUNTRY_PREFIX = "India_"
print(CLASS_NAMES)


## CELL 4 — Filter `India_` files into `data/road` + local SSD copy

Copies to Drive (`data/road`) so the filtered set is in your repo, and to `/content/rdd_india_yolo` so training is not bottlenecked by Drive I/O.


In [ ]:
def is_india(path: Path) -> bool:
    name = path.name
    return name.startswith(COUNTRY_PREFIX) or name.lower().startswith("india_")


def ensure_split_dirs(root: Path) -> None:
    for split in ("train", "val", "test"):
        (root / split / "images").mkdir(parents=True, exist_ok=True)
        (root / split / "labels").mkdir(parents=True, exist_ok=True)


ensure_split_dirs(ROAD_OUT)
ensure_split_dirs(LOCAL_INDIA)

stats = {s: {"images": 0, "labels": 0, "boxes": Counter(), "dropped_cls": Counter()} for s in ("train", "val", "test")}


for split in ("train", "val", "test"):
    src_img = RDD_SPLIT / split / "images"
    src_lbl = RDD_SPLIT / split / "labels"
    if not src_img.exists():
        print("missing", src_img)
        continue
    for img in src_img.iterdir():
        if not img.is_file() or img.suffix.lower() not in IMAGE_EXTS:
            continue
        if not is_india(img):
            continue
        lbl = src_lbl / f"{img.stem}.txt"
        keep_lines = []
        if lbl.exists():
            for line in lbl.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    continue
                cls = int(float(parts[0]))
                if cls not in KEEP_CLASSES:
                    stats[split]["dropped_cls"][cls] += 1
                    continue
                keep_lines.append(" ".join(parts[:5]))
                stats[split]["boxes"][cls] += 1
        filtered_txt = "\n".join(keep_lines) + ("\n" if keep_lines else "")
        for dest in (ROAD_OUT, LOCAL_INDIA):
            out_img = dest / split / "images" / img.name
            out_lbl = dest / split / "labels" / f"{img.stem}.txt"
            if not out_img.exists():
                shutil.copy2(img, out_img)
            out_lbl.write_text(filtered_txt, encoding="utf-8")
        stats[split]["images"] += 1
        stats[split]["labels"] += 1

for split, s in stats.items():
    print(f"=== {split} (India only) ===")
    print("images:", s["images"], "labels:", s["labels"])
    print("boxes:", dict(s["boxes"]))
    print("dropped non-target classes:", dict(s["dropped_cls"]))

assert stats["train"]["images"] > 0, (
    f"No India images found with prefix '{COUNTRY_PREFIX}'. "
    "Print a few filenames from RDD_SPLIT/train/images and adjust COUNTRY_PREFIX."
)
print("Saved filtered road set to", ROAD_OUT)
print("Local training copy", LOCAL_INDIA)


## CELL 5 — Validate India labels


In [ ]:
cls_counts = Counter()
bad_lines = []
for split in ("train", "val", "test"):
    for txt in (LOCAL_INDIA / split / "labels").glob("*.txt"):
        for i, line in enumerate(txt.read_text(encoding="utf-8").splitlines()):
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) != 5:
                bad_lines.append((txt.name, i, line, "need 5 values"))
                continue
            cls = int(float(parts[0]))
            cls_counts[cls] += 1
            if cls not in CLASS_NAMES:
                bad_lines.append((txt.name, i, line, f"unknown class {cls}"))

print("Class box counts:", dict(cls_counts))
print("Mapped:", {CLASS_NAMES.get(k, k): v for k, v in cls_counts.items()})
print("Bad lines:", len(bad_lines))
assert set(cls_counts).issubset({0, 1, 2, 3})
print("OK")


## CELL 6 — Merge crack dataset from `data/` (skip `data/road`)

Order:
1. YOLO boxes under `data/crack` (or other `images/`+`labels/` under `data/`, except `road/`).
2. If that is empty, convert the segmentation set at `municipal_ai_system/data/Crack dataset` (images + masks) into YOLO boxes and also write them to `data/crack`. Untyped pavement cracks are mapped to class **0** (`D00_longitudinal_crack`). Files matching `noncrack*` are skipped.


In [ ]:
def find_yolo_splits(root: Path) -> dict[str, tuple[Path, Path]]:
    found = {}
    if not root.exists():
        return found
    for split in ("train", "val", "test"):
        direct_img = root / split / "images"
        direct_lbl = root / split / "labels"
        if direct_img.exists() and direct_lbl.exists():
            found[split] = (direct_img, direct_lbl)
            continue
        for img_dir in root.rglob("images"):
            if not img_dir.is_dir():
                continue
            parts = {p.lower() for p in img_dir.parts}
            if "road" in parts:
                continue
            if split not in parts and split != "train":
                continue
            if split == "train" and ({"val", "test"} & parts):
                continue
            lbl_dir = img_dir.parent / "labels"
            if lbl_dir.exists():
                found[split] = (img_dir, lbl_dir)
                break
    return found


def copy_split(src_img: Path, src_lbl: Path, dest_root: Path, split: str, prefix: str = "") -> int:
    n = 0
    dst_i = dest_root / split / "images"
    dst_l = dest_root / split / "labels"
    dst_i.mkdir(parents=True, exist_ok=True)
    dst_l.mkdir(parents=True, exist_ok=True)
    for img in src_img.iterdir():
        if not img.is_file() or img.suffix.lower() not in IMAGE_EXTS:
            continue
        name = f"{prefix}{img.name}" if prefix else img.name
        stem = Path(name).stem
        shutil.copy2(img, dst_i / name)
        src_txt = src_lbl / f"{img.stem}.txt"
        dst_txt = dst_l / f"{stem}.txt"
        if src_txt.exists():
            shutil.copy2(src_txt, dst_txt)
        else:
            dst_txt.write_text("", encoding="utf-8")
        n += 1
    return n


for split in ("train", "val", "test"):
    (LOCAL_COMBINED / split / "images").mkdir(parents=True, exist_ok=True)
    (LOCAL_COMBINED / split / "labels").mkdir(parents=True, exist_ok=True)

india_copied = 0
for split in ("train", "val", "test"):
    india_copied += copy_split(
        LOCAL_INDIA / split / "images",
        LOCAL_INDIA / split / "labels",
        LOCAL_COMBINED,
        split,
        prefix="",
    )
print("Copied India images into combined:", india_copied)


def find_mask_splits(root: Path) -> dict[str, tuple[Path, Path]]:
    found = {}
    if not root.exists():
        return found
    for split, aliases in (("train", ("train",)), ("val", ("val", "valid")), ("test", ("test",))):
        for alias in aliases:
            for img_name, mask_name in (("images", "masks"),):
                img_dir = root / alias / img_name
                mask_dir = root / alias / mask_name
                if img_dir.is_dir() and mask_dir.is_dir():
                    found[split] = (img_dir, mask_dir)
                    break
            if split in found:
                break
        if split not in found:
            img_dir = root / "images"
            mask_dir = root / "masks"
            if split == "train" and img_dir.is_dir() and mask_dir.is_dir():
                found["train"] = (img_dir, mask_dir)
    return found


def mask_to_yolo_lines(mask_path: Path, class_id: int = 0, min_area: int = 40) -> list[str]:
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    _, bw = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h, w = mask.shape[:2]
    lines = []
    for contour in contours:
        x, y, bw_, bh_ = cv2.boundingRect(contour)
        if bw_ * bh_ < min_area:
            continue
        xc = (x + bw_ / 2) / w
        yc = (y + bh_ / 2) / h
        nw = bw_ / w
        nh = bh_ / h
        lines.append(f"{class_id} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
    return lines


def convert_seg_to_yolo(src_img: Path, src_mask: Path, dest_root: Path, split: str, prefix: str) -> int:
    n = 0
    dst_i = dest_root / split / "images"
    dst_l = dest_root / split / "labels"
    dst_i.mkdir(parents=True, exist_ok=True)
    dst_l.mkdir(parents=True, exist_ok=True)
    mask_index = {p.stem.lower(): p for p in src_mask.iterdir() if p.is_file()}
    for img in src_img.iterdir():
        if not img.is_file() or img.suffix.lower() not in IMAGE_EXTS:
            continue
        if img.name.lower().startswith("noncrack"):
            continue
        mask = mask_index.get(img.stem.lower())
        if mask is None:
            for key, path in mask_index.items():
                if key.startswith(img.stem.lower()) or img.stem.lower().startswith(key):
                    mask = path
                    break
        name = f"{prefix}{img.name}"
        stem = Path(name).stem
        shutil.copy2(img, dst_i / name)
        lines = mask_to_yolo_lines(mask) if mask is not None else []
        (dst_l / f"{stem}.txt").write_text(
            "\n".join(lines) + ("\n" if lines else ""),
            encoding="utf-8",
        )
        n += 1
    return n


def yolo_splits_excluding_road(root: Path) -> dict[str, tuple[Path, Path]]:
    cleaned = {}
    for split, (img_dir, lbl_dir) in find_yolo_splits(root).items():
        parts = {p.lower() for p in img_dir.parts}
        if "road" in parts:
            continue
        cleaned[split] = (img_dir, lbl_dir)
    return cleaned


crack_splits = yolo_splits_excluding_road(CRACK_ROOT)
if not crack_splits:
    crack_splits = yolo_splits_excluding_road(DATA_ROOT)

crack_n = 0
for split, (img_dir, lbl_dir) in crack_splits.items():
    added = copy_split(img_dir, lbl_dir, LOCAL_COMBINED, split, prefix="crack_")
    crack_n += added
    print(f"crack YOLO {split}: +{added} from {img_dir}")

if crack_n == 0:
    mask_splits = find_mask_splits(CRACK_SEG)
    if not mask_splits:
        mask_splits = find_mask_splits(CRACK_ROOT)
    if "val" not in mask_splits and "test" in mask_splits:
        mask_splits["val"] = mask_splits["test"]
    for split, (img_dir, mask_dir) in mask_splits.items():
        yolo_split = "val" if split == "test" and "val" not in mask_splits else split
        added = convert_seg_to_yolo(
            img_dir,
            mask_dir,
            LOCAL_COMBINED,
            yolo_split,
            prefix="crackseg_",
        )
        crack_n += added
        print(f"crack SEG {split} → combined/{yolo_split}: +{added} from {img_dir}")

if crack_n == 0:
    print("No extra crack set found — training on India road images only.")
    print("Put YOLO data in data/crack or upload Crack dataset images/masks on Drive.")
else:
    print("Merged crack images:", crack_n)

for split in ("train", "val", "test"):
    n_img = len(list((LOCAL_COMBINED / split / "images").glob("*")))
    n_lbl = len(list((LOCAL_COMBINED / split / "labels").glob("*.txt")))
    print(f"combined {split}: images={n_img} labels={n_lbl}")


## CELL 7 — dataset.yaml (train on local combined set)


In [ ]:
data_yaml = {
    "path": str(LOCAL_COMBINED),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": CLASS_NAMES,
    "nc": 4,
}
yaml_path = LOCAL_COMBINED / "data.yaml"
yaml_path.write_text(yaml.dump(data_yaml, sort_keys=False), encoding="utf-8")
(ROAD_OUT / "data.yaml").write_text(
    yaml.dump(
        {
            "path": str(ROAD_OUT),
            "train": "train/images",
            "val": "val/images",
            "test": "test/images",
            "names": CLASS_NAMES,
            "nc": 4,
        },
        sort_keys=False,
    ),
    encoding="utf-8",
)
print(yaml_path.read_text())
print("Also wrote", ROAD_OUT / "data.yaml")


## CELL 8 — Preview boxes


In [ ]:
def draw_yolo_boxes(img_path: Path, lbl_path: Path):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    h, w = img.shape[:2]
    vis = img.copy()
    colors = [(0, 255, 0), (255, 0, 0), (0, 165, 255), (0, 0, 255)]
    text = lbl_path.read_text(encoding="utf-8").strip()
    for line in text.splitlines():
        cls, xc, yc, nw, nh = [float(x) for x in line.split()]
        cls = int(cls)
        x1 = int((xc - nw / 2) * w)
        y1 = int((yc - nh / 2) * h)
        x2 = int((xc + nw / 2) * w)
        y2 = int((yc + nh / 2) * h)
        cv2.rectangle(vis, (x1, y1), (x2, y2), colors[cls % 4], 2)
        cv2.putText(
            vis,
            ID_TO_CLASS.get(cls, str(cls)),
            (x1, max(15, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            colors[cls % 4],
            2,
        )
    return cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

imgs = sorted((LOCAL_COMBINED / "train" / "images").glob("*"))[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_path in zip(axes.ravel(), imgs):
    lbl = LOCAL_COMBINED / "train" / "labels" / f"{img_path.stem}.txt"
    vis = draw_yolo_boxes(img_path, lbl) if lbl.exists() else None
    ax.imshow(vis if vis is not None else np.zeros((64, 64, 3), np.uint8))
    ax.set_title(img_path.name[:28], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


## CELL 9 — Train (fast T4 defaults)

`yolov8n` + `amp=True` + local `/content` data is the quickest path on the free T4. Set `FAST_SMOKE = True` for a 3-epoch path check, then `False` for a real run.


In [ ]:
FAST_SMOKE = False  # True → 3 epochs only
DEVICE = 0 if torch.cuda.is_available() else "cpu"
PRETRAINED = "yolov8n.pt"  # switch to yolov8s.pt if you have spare GPU time
TRAIN_EPOCHS = 3 if FAST_SMOKE else 20
IMG_SIZE = 640
BATCH = 16 if torch.cuda.is_available() else 8
PROJECT = "/content/runs/rdd_india"
RUN_NAME = "yolov8n_india"

model = YOLO(PRETRAINED)
train_results = model.train(
    data=str(yaml_path),
    epochs=TRAIN_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    workers=2,
    patience=8 if FAST_SMOKE else 12,
    amp=True,
    seed=RANDOM_SEED,
    cache=False,
    plots=True,
)
print(train_results)

best_pt = Path(PROJECT) / RUN_NAME / "weights" / "best.pt"
if not best_pt.exists():
    found = list(Path("/content").rglob("best.pt"))
    best_pt = found[0] if found else best_pt
print("best.pt =", best_pt)

AI_MODELS.mkdir(parents=True, exist_ok=True)
drive_models = MYDRIVE / "models" / "yolov8_india"
drive_models.mkdir(parents=True, exist_ok=True)
(ROAD_OUT / "weights").mkdir(parents=True, exist_ok=True)

for dest in (
    AI_MODELS / "best.pt",
    drive_models / "best.pt",
    ROAD_OUT / "weights" / "best.pt",
):
    shutil.copy2(best_pt, dest)
    print("Saved:", dest)


## CELL 10 — Validation


In [ ]:
eval_model = YOLO(str(best_pt))
m = eval_model.val(data=str(yaml_path), split="val", device=DEVICE, plots=True)
print("Precision:", float(m.box.mp))
print("Recall:", float(m.box.mr))
print("mAP50:", float(m.box.map50))
print("mAP50-95:", float(m.box.map))


## CELL 11 — Upload an image and print a defect report


In [ ]:
SEVERITY_TEXT = {
    "LOW": "LOW — Monitor",
    "MODERATE": "MODERATE — Maintenance Recommended",
    "HIGH": "HIGH — Immediate Attention",
}

def box_area_pct(xyxy, img_w, img_h):
    x1, y1, x2, y2 = xyxy
    return ((x2 - x1) * (y2 - y1)) / (img_w * img_h) * 100.0

def severity_from_area(area_pct, class_id, conf):
    if class_id == 3 and (area_pct >= 1.5 or conf >= 0.70):
        return "HIGH"
    if area_pct >= 5.0:
        return "HIGH"
    if area_pct >= 1.0:
        return "MODERATE"
    return "LOW"

def overall_severity(levels):
    order = {"LOW": 0, "MODERATE": 1, "HIGH": 2}
    return max(levels, key=lambda s: order[s]) if levels else "LOW"

def format_report(dets, img_w, img_h):
    enriched = []
    for d in dets:
        area = box_area_pct(d["xyxy"], img_w, img_h)
        sev = severity_from_area(area, d["class_id"], d["confidence"])
        enriched.append({**d, "area_pct": area, "severity": sev})
    counts = {name: 0 for name in CLASS_NAMES.values()}
    for e in enriched:
        counts[CLASS_NAMES[e["class_id"]]] += 1
    overall = overall_severity([e["severity"] for e in enriched])
    lines = [
        f"Total Defects Found : {len(enriched)}",
        f" Overall Severity    : {SEVERITY_TEXT[overall]}",
        "",
        " --- Defects Breakdown by Class ---",
    ]
    for name in CLASS_NAMES.values():
        lines.append(f"   • {name:<24} : {counts[name]}")
    lines += ["", " --- Detailed Detection List ---"]
    if not enriched:
        lines.append("  (none)")
    else:
        for i, e in enumerate(enriched, 1):
            x1, y1, x2, y2 = e["xyxy"]
            lines += [
                f"  [{i}] Class Name  : {CLASS_NAMES[e['class_id']]}",
                f"      Confidence  : {e['confidence']*100:.1f}%",
                f"      Severity    : {e['severity']} (Surface Area: {e['area_pct']:.2f}%)",
                f"      BoundingBox : [{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]",
                "",
            ]
    return "\n".join(lines)

from google.colab import files
from IPython.display import Image as IPImage, display

pred_model = YOLO(str(best_pt))
up = files.upload()
img_path = list(up.keys())[0]
img = cv2.imread(img_path)
h, w = img.shape[:2]
res = pred_model.predict(img_path, conf=0.25, verbose=False)[0]
dets = []
if res.boxes is not None:
    for b in res.boxes:
        dets.append({
            "class_id": int(b.cls.item()),
            "confidence": float(b.conf.item()),
            "xyxy": [float(x) for x in b.xyxy[0].tolist()],
        })
print(format_report(dets, w, h))
out = Path("/content/pred_india.jpg")
cv2.imwrite(str(out), res.plot())
display(IPImage(str(out)))


## CELL 12 — Upload a video (track + download)


In [ ]:
up = files.upload()
video_path = list(up.keys())[0]
best_by_track = {}
frame_size = None
writer = None
out_mp4 = Path("/content/pred_india_video.mp4")

for r in pred_model.track(source=video_path, conf=0.25, persist=True, stream=True, verbose=False):
    if frame_size is None:
        frame_size = (r.orig_shape[1], r.orig_shape[0])
        writer = cv2.VideoWriter(str(out_mp4), cv2.VideoWriter_fourcc(*"mp4v"), 20.0, frame_size)
    writer.write(r.plot())
    if r.boxes is None or r.boxes.id is None:
        continue
    for b, tid in zip(r.boxes, r.boxes.id.cpu().tolist()):
        tid = int(tid)
        conf_v = float(b.conf.item())
        det = {
            "class_id": int(b.cls.item()),
            "confidence": conf_v,
            "xyxy": [float(x) for x in b.xyxy[0].tolist()],
        }
        if tid not in best_by_track or conf_v > best_by_track[tid]["confidence"]:
            best_by_track[tid] = det

if writer:
    writer.release()
w, h = frame_size
print(format_report(list(best_by_track.values()), w, h))
files.download(str(out_mp4))
